# Hippocampal-Entorhinal Network Eigenmode Analysis — Corrected Schur Decomposition

This corrected notebook implements the technical conventions requested:

1. The original CSV is interpreted as **rows = outgoing/source** and **columns = receivers/targets**.
2. The state-update matrix for $x_{t+1}=Mx_t$ is therefore `M = df_source_receiver.to_numpy().T`.
3. The Schur decomposition remains in the standard upper-triangular form $M = Q T Q^H$.
4. In Schur coordinates, $y_{t+1}=Ty$, so $T_{ij}$ means **source mode $j$ drives receiver mode $i$**.
5. The printed cascade order lists upstream drivers first by traversing Schur indexes from highest to lowest.
6. Discrete-time stability and transient growth are analyzed using $ho(M)$, $M^k$, and a discrete resolvent scan, not $e^{Mt}$.
7. True eigenvectors are computed with `scipy.linalg.eig`; Schur vectors are not mislabeled as eigenvectors.


## 1. Library Imports & Data Loading
First, we load the required scientific computing libraries and the connectivity matrix.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the matrix.
# CSV convention:
#   rows    = outgoing/source cell types
#   columns = receiving/target cell types
csv_path = Path("mij_matrix.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "Could not find mij_matrix.csv. Place it in the same directory as this notebook or update csv_path."
    )

df_source_receiver = pd.read_csv(csv_path, index_col=0)
df = df_source_receiver  # backwards-compatible name used later

print(f"Loaded source→receiver connectivity matrix of shape: {df_source_receiver.shape}")
print("CSV convention: rows = outgoing/source, columns = receivers/targets")
df_source_receiver.head()


## 2. Matrix Orientation and Normalization

Your original `mij_matrix.csv` uses:

\[
\text{row}=\text{outgoing/source}, \qquad \text{column}=\text{receiver/target}.
\]

For the discrete-time state model

\[
x_{t+1}=Mx_t,
\]

the standard convention is:

\[
M_{ij}=\text{effect of source }j\text{ on receiver }i.
\]

So the analysis matrix is the transpose of the CSV:

\[
M = \texttt{df\_source\_receiver.to\_numpy().T}.
\]

This keeps the usual column-vector update convention and keeps the Schur factorization in the standard upper-triangular form:

\[
M = Q T Q^H.
\]

In Schur coordinates,

\[
y_{t+1}=Ty,\qquad x=Qy.
\]

Because $T_{ij}$ means **mode $j$ drives mode $i$**, higher-index Schur modes can drive lower-index modes. The notebook prints higher-index modes first when listing upstream drivers.


In [ ]:
# ── Normalization control ─────────────────────────────────────────────────
NORMALIZATION = "none"          # "none", "row_l1", or "spectral_radius"
TARGET_SPECTRAL_RADIUS = 1.0    # only used for spectral_radius

# CSV matrix: rows = sources/outgoing, columns = receivers/targets
matrix_source_receiver = df_source_receiver.to_numpy(dtype=float)

# State-update matrix for x[t+1] = M @ x[t].
# Standard convention: M[receiver, source].
M_raw = matrix_source_receiver.T.copy()

# Labels in M order. After transposition, M rows are receivers.
cell_labels = list(df_source_receiver.columns)

if NORMALIZATION == "none":
    M = M_raw.copy()
    norm_label = "No normalization"
    print("Method: None")

elif NORMALIZATION == "row_l1":
    # For x[t+1] = M x[t], column normalization preserves per-source outgoing proportions.
    col_abs_sums = np.sum(np.abs(M_raw), axis=0)
    col_abs_sums[col_abs_sums == 0] = 1.0
    M = M_raw / col_abs_sums[np.newaxis, :]
    norm_label = "Column L1 normalization after orientation correction"
    print("Method: Column L1 normalization")
    print("Column |L1| sums (first 10):", np.sum(np.abs(M), axis=0)[:10].round(6))

elif NORMALIZATION == "spectral_radius":
    eigvals_raw = np.linalg.eigvals(M_raw)
    rho = np.max(np.abs(eigvals_raw))
    if rho == 0:
        raise ValueError("Cannot spectral-radius normalize a matrix with spectral radius 0.")
    M = M_raw * (TARGET_SPECTRAL_RADIUS / rho)
    norm_label = f"Spectral radius normalization (target ρ = {TARGET_SPECTRAL_RADIUS})"
    print("Method: Spectral radius normalization")

else:
    raise ValueError("NORMALIZATION must be 'none', 'row_l1', or 'spectral_radius'.")

eigenvalues_check = np.linalg.eigvals(M)
rho_check = np.max(np.abs(eigenvalues_check))

print(f"Matrix shape: {M.shape}")
print(f"Normalization: {norm_label}")
print(f"Discrete-time spectral radius ρ(M): {rho_check:.6f}")
print(f"Discrete-time stability: {'asymptotically stable' if rho_check < 1 else 'not asymptotically stable'}")


## 3. Schur Decomposition

We compute the complex Schur form:

\[
M = Q T Q^H.
\]

Here:

- `T` is upper triangular.
- `Q` is unitary.
- `diag(T)` contains the eigenvalues.
- Columns of `Q` are **Schur vectors**, not ordinary eigenvectors.

For the discrete-time model,

\[
y_{t+1}=Ty.
\]

Since $T_{ij}$ maps source Schur coordinate $j$ into receiver Schur coordinate $i$, higher-index modes are the natural upstream drivers in an upper-triangular Schur form.


In [ ]:
    from scipy import linalg

    # Complex Schur decomposition: M = Q T Q^H
    # T: upper triangular, eigenvalues on diagonal
    # Q: unitary Schur vectors
    T_schur, Q = linalg.schur(M, output="complex")

    # Do not independently flip/scale columns of Q unless T is transformed too.
    eigenvalues = np.diag(T_schur)

    # Reference eigendecomposition for true eigenvectors when needed.
    eigvals_ref, eigvecs_ref = linalg.eig(M)

    # Validation
    reconstruction_rel_error = (
        np.linalg.norm(M - Q @ T_schur @ Q.conj().T, ord="fro")
        / max(np.linalg.norm(M, ord="fro"), np.finfo(float).eps)
    )
    unitarity_error = np.linalg.norm(Q.conj().T @ Q - np.eye(Q.shape[0]), ord="fro")
    upper_triangular_error = np.linalg.norm(np.tril(T_schur, k=-1), ord="fro")
    cond_Q = np.linalg.cond(Q)
    cond_eigvecs = np.linalg.cond(eigvecs_ref)

    comm = M @ M.conj().T - M.conj().T @ M
    non_normality = np.linalg.norm(comm, "fro")

    print("Schur decomposition complete.")
    print(f"  T shape: {T_schur.shape}  (upper triangular)")
    print(f"  Q shape: {Q.shape}  (unitary Schur vectors)")
    print(f"  reconstruction relative error: {reconstruction_rel_error:.3e}")
    print(f"  unitarity error ||QᴴQ-I||_F: {unitarity_error:.3e}")
    print(f"  lower-triangle norm ||tril(T,-1)||_F: {upper_triangular_error:.3e}")
    print(f"  cond(Q): {cond_Q:.6f}")
    print(f"  spectral radius from diag(T): {np.max(np.abs(eigenvalues)):.6f}")

    print(f"
Non-normality ||MMᴴ - MᴴM||_F: {non_normality:.4e}")
    print("
Condition number comparison:")
    print(f"  Eigenvector matrix: {cond_eigvecs:.4e}")
    print(f"  Schur Q matrix:     {cond_Q:.6f}")
    print(f"  Ratio: {cond_eigvecs / cond_Q:.2e}x worse for eigenvectors")


## 4. Visualizing the Eigenvalue Spectrum
Eigenvalues come from `diag(T_schur)` — identical to those from standard eigendecomposition, but now paired with orthonormal Schur vectors rather than potentially ill-conditioned eigenvectors.
We plot on the complex plane: Real Part = growth/decay rate, Imaginary Part = oscillation frequency.

In [ ]:
plt.figure(figsize=(12, 10))
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.scatter(eigenvalues.real, eigenvalues.imag, color="forestgreen", alpha=0.7, edgecolors="k", s=60)

# Unit circle for discrete-time stability
theta = np.linspace(0, 2*np.pi, 400)
plt.plot(np.cos(theta), np.sin(theta), linestyle="--", linewidth=1.0, label="Unit circle")

# Annotate largest-magnitude modes
idx_sorted_by_radius = np.argsort(np.abs(eigenvalues))[::-1]
annotated_count = 0
for i in idx_sorted_by_radius:
    val = eigenvalues[i]
    if val.imag < -1e-5:
        continue
    if annotated_count < 8:
        plt.annotate(
            f"Schur index {i+1}\n({val.real:.3f} + {val.imag:.3f}j)",
            (val.real, val.imag),
            textcoords="offset points",
            xytext=(15, 15 if val.imag >= 0 else -25),
            ha="center",
            fontsize=9,
            arrowprops=dict(arrowstyle="->", color="red", lw=0.8),
        )
        annotated_count += 1

plt.title(f"Eigenvalue Spectrum for Discrete-Time M ({norm_label})", fontsize=14, fontweight="bold")
plt.xlabel("Real Part", fontsize=12)
plt.ylabel("Imaginary Part", fontsize=12)
plt.grid(True, which="both", linestyle=":", alpha=0.5)
plt.axis("equal")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Mode Listing: Upstream Drivers First

The listing below uses the standard upper-triangular Schur convention and prints modes in reverse Schur index order:

\[
n, n-1, \ldots, 1.
\]

For $y_{t+1}=Ty$:

- $T_{ij}$ means mode $j$ drives mode $i$.
- Incoming sources to mode $i$ are found across row $i$: `T_schur[i, j]`.
- Outgoing targets from mode $i$ are found down column $i$: `T_schur[k, i]`.
- In an upper-triangular $T$, higher-index modes drive lower-index modes.


In [ ]:
# --- Schur-mode listing: upstream drivers first ---

eigvals_left, eigvecs_left = linalg.eig(M, left=True, right=False)

def format_complex(z, tol=1e-8):
    z = complex(z)
    if abs(z.imag) < tol:
        return f"{z.real:.4f}"
    sign = "+" if z.imag >= 0 else "-"
    return f"{z.real:.4f} {sign} {abs(z.imag):.4f}i"

def format_schur_coupling(coupling):
    return f"coupling T = {format_complex(coupling)} | |coupling| = {abs(coupling):.4f}"

def get_relative_phase_str(w, dominant_phase, is_oscillatory):
    if not is_oscillatory:
        return "N/A"
    diff = np.angle(w) - dominant_phase
    diff_wrapped = np.arctan2(np.sin(diff), np.cos(diff))
    diff_deg = np.degrees(diff_wrapped)
    if abs(diff_deg) < 1e-2:
        return "0.0° (Ref)"
    return f"{diff_deg:+.1f}°"

def build_unique_modes_upstream_first(eigenvalues, Q, eigvals_left, eigvecs_left, tol=1e-5):
    seen = set()
    schur_order = np.arange(len(eigenvalues) - 1, -1, -1)
    unique_modes = []
    schur_to_printed = {}

    for i in schur_order:
        if i in seen:
            continue

        val = eigenvalues[i]
        match_idx = np.argmin(np.abs(eigvals_left - val))
        left_val = eigvals_left[match_idx]
        left_vec = eigvecs_left[:, match_idx]

        if abs(val.imag) < tol:
            unique_modes.append(("Real", val, Q[:, i], left_val, left_vec, i, [i]))
            schur_to_printed[i] = len(unique_modes)
            seen.add(i)
        else:
            conjugate_j = None
            for j in schur_order:
                if j != i and j not in seen:
                    val2 = eigenvalues[j]
                    if abs(val.real - val2.real) < tol and abs(val.imag + val2.imag) < tol:
                        conjugate_j = j
                        break

            if conjugate_j is None:
                unique_modes.append(("Complex", val, Q[:, i], left_val, left_vec, i, [i]))
                schur_to_printed[i] = len(unique_modes)
                seen.add(i)
            else:
                unique_modes.append(("Complex Pair", val, Q[:, i], left_val, left_vec, i, [i, conjugate_j]))
                schur_to_printed[i] = len(unique_modes)
                schur_to_printed[conjugate_j] = len(unique_modes)
                seen.add(i)
                seen.add(conjugate_j)

    return unique_modes, schur_to_printed

unique_modes, schur_to_printed = build_unique_modes_upstream_first(
    eigenvalues, Q, eigvals_left, eigvecs_left
)

print(f"Identified {len(unique_modes)} unique Schur modes.")
print("Printed order: upstream drivers first, using original Schur indexes n → 1.")
print("Convention: T[row receiver mode, column source mode].")

for printed_idx, (mtype, val, vec, left_val, left_vec, orig_idx, member_indices) in enumerate(unique_modes, start=1):
    is_oscillatory = mtype != "Real"
    print("=" * 80)

    if mtype == "Real":
        dyn_desc = "Non-oscillatory discrete-time multiplier."
    else:
        dyn_desc = "Oscillatory/discrete spiral mode from a complex pair."

    print(
        f"Cascade Step {printed_idx} "
        f"[Schur index {orig_idx+1}; paired indexes {[m+1 for m in member_indices]}] "
        f"| Type = {mtype} | λ = {format_complex(val)} | |λ| = {abs(val):.4f}"
    )
    print(f"Dynamics: {dyn_desc}")
    print(f"Left eigenvalue matched to this Schur diagonal value: {format_complex(left_val)}")

    # Incoming sources: row orig_idx, columns j.
    incoming_candidates = []
    for j in range(T_schur.shape[1]):
        if j == orig_idx:
            continue
        coupling_val = T_schur[orig_idx, j]
        if abs(coupling_val) > 1e-5:
            source_step = schur_to_printed.get(j)
            if source_step is not None and source_step != printed_idx:
                incoming_candidates.append((abs(coupling_val), coupling_val, source_step, j))

    incoming_candidates.sort(key=lambda x: x[0], reverse=True)
    unique_incoming = []
    seen_source_steps = set()
    for _, c_val, source_step, source_schur_idx in incoming_candidates:
        if source_step not in seen_source_steps:
            seen_source_steps.add(source_step)
            unique_incoming.append((c_val, source_step, source_schur_idx))
            if len(unique_incoming) == 3:
                break

    if unique_incoming:
        parts = [
            f"Cascade Step {step} / Schur {schur_idx+1} "
            f"(T[receiver Schur {orig_idx+1}, source Schur {schur_idx+1}]; {format_schur_coupling(c_val)})"
            for c_val, step, schur_idx in unique_incoming
        ]
        print("Top incoming source(s): " + " | ".join(parts))
    else:
        print("Top incoming source(s): None under upper-triangular Schur ordering")

    # Outgoing targets: column orig_idx, rows k.
    outgoing_candidates = []
    for k in range(T_schur.shape[0]):
        if k == orig_idx:
            continue
        coupling_val = T_schur[k, orig_idx]
        if abs(coupling_val) > 1e-5:
            target_step = schur_to_printed.get(k)
            if target_step is not None and target_step != printed_idx:
                outgoing_candidates.append((abs(coupling_val), coupling_val, target_step, k))

    outgoing_candidates.sort(key=lambda x: x[0], reverse=True)
    unique_outgoing = []
    seen_target_steps = set()
    for _, c_val, target_step, target_schur_idx in outgoing_candidates:
        if target_step not in seen_target_steps:
            seen_target_steps.add(target_step)
            unique_outgoing.append((c_val, target_step, target_schur_idx))
            if len(unique_outgoing) == 3:
                break

    if unique_outgoing:
        parts = [
            f"Cascade Step {step} / Schur {schur_idx+1} "
            f"(T[receiver Schur {schur_idx+1}, source Schur {orig_idx+1}]; {format_schur_coupling(c_val)})"
            for c_val, step, schur_idx in unique_outgoing
        ]
        print("Top downstream target(s): " + " | ".join(parts))
    else:
        print("Top downstream target(s): None")

    print("Principal contributing cell types (Schur vector Q[:, index]):")
    dominant_idx_schur = np.argsort(np.abs(vec))[::-1][0]
    dominant_phase_schur = np.angle(vec[dominant_idx_schur])

    for tidx in np.argsort(np.abs(vec))[::-1][:5]:
        cell_name = cell_labels[tidx]
        w = vec[tidx]
        energy_pct = abs(w) ** 2 * 100
        phase_str = get_relative_phase_str(w, dominant_phase_schur, is_oscillatory)
        print(
            f"  - {cell_name:<45} | weight = {format_complex(w):>18} "
            f"| energy = {energy_pct:5.1f}% | phase = {phase_str}"
        )

    print("Principal contributing cell types (left eigenvector; action/readout/sensitivity):")
    dominant_idx_left = np.argsort(np.abs(left_vec))[::-1][0]
    dominant_phase_left = np.angle(left_vec[dominant_idx_left])

    for tidx in np.argsort(np.abs(left_vec))[::-1][:5]:
        cell_name = cell_labels[tidx]
        w_left = left_vec[tidx]
        energy_pct_left = abs(w_left) ** 2 * 100
        phase_str_left = get_relative_phase_str(w_left, dominant_phase_left, is_oscillatory)
        print(
            f"  - {cell_name:<45} | weight = {format_complex(w_left):>18} "
            f"| energy = {energy_pct_left:5.1f}% | phase = {phase_str_left}"
        )



## 6. Dominant Eigenvalue and True Dominant Eigenvector

Use eigendecomposition when you specifically need eigenvectors.

Schur vectors are orthonormal and numerically stable, but `Q[:, i]` is a Schur vector, not generally an eigenvector. The block below therefore uses `eigvecs_ref` for the dominant eigenvector and uses `Q` only when discussing Schur coordinates.


In [ ]:
# Dominant eigenvalue/eigenvector from true eigendecomposition
dominant_index_eig = np.argmax(np.abs(eigvals_ref))
dominant_eigenvalue = eigvals_ref[dominant_index_eig]
dominant_eigenvector = eigvecs_ref[:, dominant_index_eig]

# Closest Schur diagonal index for reporting
dominant_index_schur = np.argmin(np.abs(eigenvalues - dominant_eigenvalue))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: eigenvalue spectrum
ax1.scatter(eigenvalues.real, eigenvalues.imag, c="blue", alpha=0.6, label="Eigenvalues")
ax1.scatter(
    dominant_eigenvalue.real,
    dominant_eigenvalue.imag,
    c="red",
    s=100,
    edgecolors="black",
    zorder=5,
    label="Dominant eigenvalue",
)
unit_circle = plt.Circle((0, 0), 1, color="gray", fill=False, linestyle="--", label="Unit circle")
ax1.add_artist(unit_circle)
ax1.set_xlabel("Real Part")
ax1.set_ylabel("Imaginary Part")
ax1.set_title("Eigenvalue Spectrum of the Discrete-Time Connectivity Matrix")
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.axvline(0, color="gray", linewidth=0.5)
ax1.axhline(0, color="gray", linewidth=0.5)
ax1.set_aspect("equal", adjustable="box")
ax1.legend()

# Plot 2: true dominant eigenvector components
eigenvector_df = pd.DataFrame({
    "Cell type": cell_labels,
    "Magnitude": np.abs(dominant_eigenvector),
}).sort_values("Magnitude", ascending=False).head(20)

ax2.barh(eigenvector_df["Cell type"], eigenvector_df["Magnitude"])
ax2.set_xlabel("Magnitude")
ax2.set_title("Top 20 Contributions to the True Dominant Eigenvector")
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Dominant eigenvalue: {format_complex(dominant_eigenvalue)}")
print(f"Spectral radius ρ(M): {abs(dominant_eigenvalue):.6f}")
print(f"Nearest Schur diagonal index: {dominant_index_schur + 1}")
print(
    "Discrete-time stability: "
    + ("asymptotically stable because ρ(M) < 1" if abs(dominant_eigenvalue) < 1 else "not asymptotically stable because ρ(M) >= 1")
)


## 1. Eigenvalue Spectrum and Network Stability

The plot on the left shows all the eigenvalues of your network on the complex plane.

- **Dominant Eigenvalue:** The single red dot on the far right is the dominant eigenvalue. Its value is approximately 26,213.

- **Biological Meaning (Stability):** The magnitude of the dominant eigenvalue, known as the spectral radius, dictates the stability of the network. A value greater than 1.0 means that, without any balancing inhibition, activity in the network would grow uncontrollably, leading to a state of runaway excitation similar to an epileptic seizure. The massive value here is expected for a purely excitatory matrix. Biologically, this highlights the absolute necessity of inhibitory neurons, which are not fully captured in this matrix, to maintain network stability and keep the effective spectral radius near the critical value of 1.0, where complex computations can occur. STDP, in concert with inhibitory plasticity, is the mechanism that tunes synaptic weights to achieve this delicate balance.

## 2. Dominant Eigenvector and the Principal Network Mode

The bar chart on the right shows which neuron types are the most significant components of the dominant eigenvector. This eigenvector represents the network's "principal mode"—the most stable and influential pattern of activity.

- **Key Contributors:** The pattern is overwhelmingly dominated by neurons from the Entorhinal Cortex (EC), specifically EC LI II Multipolar Pyramidal, EC LI II Pyramidal Fan, and EC LII III Pyramidal Tripolar neurons.

- **Biological Meaning (Information Flow):** This result is highly significant. It suggests that the primary, most powerful flow of information that drives the entire hippocampal circuit originates from the superficial layers of the Entorhinal Cortex. This is the very beginning of the canonical hippocampal loop (EC -> DG -> CA3 -> CA1). Your data's principal mathematical mode correctly identifies the biological starting point of memory encoding. STDP would act to strengthen these specific feedforward pathways from the EC if they consistently predict activity in the downstream hippocampus, thus carving out this dominant eigenvector from the underlying anatomy.

# Schur Decomposition: Upper-Triangular Coupling Map

The heatmap below visualizes the real part of $T$.

For the discrete-time model,

\[
y_{t+1}=Ty.
\]

The coupling convention is:

\[
T_{ij}: \text{ source mode } j \rightarrow \text{ receiver mode } i.
\]

Because $T$ is upper triangular, higher-index modes can feed lower-index modes. The notebook therefore prints cascade steps in reverse Schur-index order when you want upstream drivers first, while leaving $T$ itself in the standard upper-triangular form.


In [ ]:
# Visualize Schur T matrix structure
import matplotlib.colors as mcolors

plt.figure(figsize=(12, 10))
T_real = T_schur.real
mask = np.tril(np.ones_like(T_real, dtype=bool), k=-1)
sns.heatmap(
    T_real,
    cmap="coolwarm",
    center=0,
    norm=mcolors.SymLogNorm(
        linthresh=1e-3,
        linscale=1.0,
        vmin=T_real.min(),
        vmax=T_real.max(),
    ),
    mask=mask,
)
plt.title("Schur T Matrix: upper triangular Re(T)", size=14)
plt.xlabel("Source Schur mode index j", size=12)
plt.ylabel("Receiver Schur mode index i", size=12)
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()


## 7. First Printed Driver Mode

The first printed cascade step is the highest-index Schur mode, not `Q[:, 0]`.

This matches the upper-triangular convention where high-index modes can drive lower-index modes.


In [ ]:
# Top contributing cells in first printed upstream-driver Schur vector
first_driver_schur_index = len(eigenvalues) - 1
first_driver_vec = Q[:, first_driver_schur_index]

station_df = pd.DataFrame({
    "Cell type": cell_labels,
    "Contribution": np.abs(first_driver_vec),
}).sort_values("Contribution", ascending=False)

print(f"Top 5 cells in first printed upstream-driver Schur vector Q[:, {first_driver_schur_index}]:")
print(station_df.head())


## 8. Manual Coupling Trace

To trace flow in Schur coordinates:

1. Pick a printed upstream driver mode.
2. Look at its **Top downstream target(s)**.
3. Continue from that target mode.
4. Use the **Top left-vector/action cell** column as the compact label for the cell type most associated with the action/readout of that mode.

Important distinction:

- The **Schur vector** column `Q[:, i]` describes the orthonormal state pattern for Schur mode `i`.
- The **left eigenvector** describes the projection/sensitivity/readout direction associated with the corresponding eigenvalue.
- In the cascade summary below, the old “Top Schur-vector cell” label is replaced by **Top left-vector/action cell** so the trace emphasizes who is most associated with the action/readout of each mode.

Do not interpret the first column of $T$ as outputs from Station 1. In $y_{t+1}=Ty$, column $j$ contains where source mode $j$ sends coupling, and row $i$ contains what drives receiver mode $i$.


In [ ]:
# Compact mode summary table: upstream drivers first
# The compact label now uses the top LEFT-vector cell rather than the top Schur-vector cell.
# This is useful when you want the cascade trace to emphasize the action/readout/sensitivity
# associated with each mode.

mode_summary_rows = []
for printed_idx, (mtype, val, vec, left_val, left_vec, orig_idx, member_indices) in enumerate(unique_modes, start=1):
    top_schur_cell = cell_labels[int(np.argmax(np.abs(vec)))]
    top_left_cell = cell_labels[int(np.argmax(np.abs(left_vec)))]

    # Count incoming couplings above threshold.
    # Incoming sources to mode i are in row i: T_schur[i, j].
    incoming_count = 0
    for j in range(T_schur.shape[1]):
        if j != orig_idx and abs(T_schur[orig_idx, j]) > 1e-5:
            incoming_count += 1

    # Outgoing targets from mode i are in column i: T_schur[k, i].
    outgoing_count = 0
    for k in range(T_schur.shape[0]):
        if k != orig_idx and abs(T_schur[k, orig_idx]) > 1e-5:
            outgoing_count += 1

    mode_summary_rows.append({
        "Cascade step": printed_idx,
        "Schur index": orig_idx + 1,
        "Type": mtype,
        "lambda": format_complex(val),
        "|lambda|": abs(val),
        "Top left-vector/action cell": top_left_cell,
        "Top Schur-state cell": top_schur_cell,
        "Incoming count": incoming_count,
        "Outgoing count": outgoing_count,
    })

mode_summary_df = pd.DataFrame(mode_summary_rows)
display(mode_summary_df.head(20))


**Strategy 1: How to Trace it Manually**
Using the printout code we built, you can follow the "waterfall" step-by-step by tracing facilitatory (excitatory) coupling terms with positive real weights.

Step 1 (Start at the Headwaters): Find a Cascade Step (like Cascade Step 1) where the Principal Contributing Cell Types (Schur vector) are dominated by EC LI II Pyramidal Fan or MEC LII Stellate cells.

Step 2 (Hop to DG): Look at that step's Top Downstream Target(s). Look for a target step that has a positive (facilitatory) weight and is dominated by DG Granule or DG Axo-axonic cells. Let's say this is Cascade Step A.

Step 3 (Hop to CA3): Go to the printout block for Cascade Step A. Look at its Top Downstream Target(s). Find a target step with a positive (facilitatory) weight dominated by CA3 Pyramidal cells. Let's say this is Cascade Step B.

Step 4 (Hop to CA1): Go to Cascade Step B. Look at its Top Downstream Target(s). Find a target step with a positive (facilitatory) weight dominated by CA1 Pyramidal cells.

This completes the physical loop!

# Investigating Self-Connections

In [ ]:
# Self-connection check using the ORIGINAL source→receiver CSV orientation

self_connections = np.diag(df_source_receiver.to_numpy(dtype=float))

# Max outgoing connection from each source to a different receiver.
arr = df_source_receiver.to_numpy(dtype=float, copy=True)
np.fill_diagonal(arr, -np.inf)
max_outgoing = arr.max(axis=1)

autocentric_neurons = []
for i, neuron in enumerate(df_source_receiver.index):
    if self_connections[i] > max_outgoing[i]:
        autocentric_neurons.append({
            "Cell type": neuron,
            "Self-connection": self_connections[i],
            "Max outgoing to other": max_outgoing[i],
        })

autocentric_df = pd.DataFrame(autocentric_neurons)

if not autocentric_df.empty:
    autocentric_df = autocentric_df.sort_values("Self-connection", ascending=True)

    plt.figure(figsize=(12, 8))
    y_pos = np.arange(len(autocentric_df))

    plt.barh(
        y_pos - 0.2,
        autocentric_df["Self-connection"],
        height=0.4,
        align="center",
        label="Self-connection",
    )
    plt.barh(
        y_pos + 0.2,
        autocentric_df["Max outgoing to other"],
        height=0.4,
        align="center",
        label="Max outgoing to other",
    )

    plt.yticks(y_pos, autocentric_df["Cell type"])
    plt.xlabel("Connection weight")
    plt.title("Cells whose self-connection exceeds every other outgoing connection")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("--- Autocentric cell types ---")
    display(autocentric_df)
else:
    print("No cell types have a self-connection larger than every other outgoing connection.")


In [ ]:
# =============================================================================
# SECTION: DISCRETE-TIME TRANSIENT AMPLIFICATION AND RESOLVENT ANALYSIS
# =============================================================================
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

def analyze_discrete_transient_dynamics(M, k_max=50):
    """
    Discrete-time analysis for x[t+1] = M x[t].

    Computes:
      G(k) = ||M^k||_2
      spectral radius rho(M)
      peak finite-horizon amplification
      a grid-based discrete Kreiss-style lower bound outside the unit circle:
          (|z| - 1) * ||(zI - M)^(-1)||_2, |z| > 1
    """
    print("=" * 80)
    print("DISCRETE-TIME TRANSIENT AMPLIFICATION ANALYSIS")
    print("=" * 80)

    eigvals = np.linalg.eigvals(M)
    rho = np.max(np.abs(eigvals))

    powers = []
    gains = []
    Mk = np.eye(M.shape[0], dtype=complex)

    for k in range(k_max + 1):
        if k == 0:
            Mk = np.eye(M.shape[0], dtype=complex)
        elif k == 1:
            Mk = M.astype(complex)
        elif k > 1:
            Mk = M.astype(complex) @ Mk

        powers.append(k)
        gains.append(np.linalg.norm(Mk, 2))

    powers = np.array(powers)
    gains = np.array(gains)

    peak_idx = int(np.argmax(gains))
    peak_gain = float(gains[peak_idx])
    peak_k = int(powers[peak_idx])

    # Discrete-time resolvent scan outside unit circle.
    rmax = max(1.25, np.max(np.abs(eigvals)) * 1.25)
    real_grid = np.linspace(-rmax, rmax, 50)
    imag_grid = np.linspace(-rmax, rmax, 50)

    kreiss_lower_bound = 0.0
    critical_z = None

    I = np.eye(M.shape[0], dtype=complex)
    for x in real_grid:
        for y in imag_grid:
            z = x + 1j * y
            abs_z = abs(z)
            if abs_z <= 1.0:
                continue
            svals = np.linalg.svd(z * I - M, compute_uv=False)
            sigma_min = np.min(svals)
            if sigma_min <= np.finfo(float).eps:
                continue
            resolvent_norm = 1.0 / sigma_min
            val = (abs_z - 1.0) * resolvent_norm
            if val > kreiss_lower_bound:
                kreiss_lower_bound = float(val)
                critical_z = z

    print("1. ASYMPTOTIC DISCRETE-TIME LANDSCAPE:")
    print(f"   - Spectral radius rho(M): {rho:.6f}")
    print(f"   - Stable iff rho(M) < 1: {'YES' if rho < 1 else 'NO'}")
    print("-" * 50)
    print("2. FINITE-HORIZON TRANSIENT GROWTH:")
    print(f"   - Peak ||M^k||_2 over k=0..{k_max}: {peak_gain:.6f}x")
    print(f"   - Peak time step k*: {peak_k}")
    print("-" * 50)
    print("3. DISCRETE RESOLVENT / KREISS-STYLE GRID BOUND:")
    print(f"   - Grid lower-bound proxy: {kreiss_lower_bound:.6f}")
    if critical_z is not None:
        print(f"   - Critical z from grid: {critical_z.real:.4f} {'+' if critical_z.imag >= 0 else '-'} {abs(critical_z.imag):.4f}i")
    print("=" * 80)

    plt.figure(figsize=(10, 6))
    plt.plot(powers, gains, marker="o", markersize=3)
    plt.xlabel("Discrete time step k")
    plt.ylabel(r"$||M^k||_2$")
    plt.title("Discrete-Time Transient Amplification")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

    return powers, gains, peak_gain, peak_k, kreiss_lower_bound

powers, growth_curve, peak_amp, peak_k, kreiss_bound = analyze_discrete_transient_dynamics(M, k_max=50)
